# DPM Engine · Common

**Purpose.** Definitions shared by the notebooks of the DPM engine (`DE_ENR_NB_LGL_REG_DPM_PARAM_LOAD`, `DE_ENR_NB_LGL_REG_DPM_METADATA_VALIDATION` and `DE_ENR_NB_LGL_REG_DPM_PROCESSING`): text and period helpers, reading of SCD snapshots, filter values, expressions and source table contract, reading of a DataLoader upload in the raw tables and retry of concurrent Delta writes. Each notebook loads it with `%run` after its parameters cell, so validation and calculation apply exactly the same rules. It is not run on its own.

**Sources:** none.

**Sink tables:** none.

**Parameters:** none. The functions that read tables use the table names of the parameters cell of the notebook that loads it.

**Changelog:** 2026-11-11 | Notebook creation (definitions moved out of the two notebooks).

2026-11-18 | Blank rows of every tab of a DataLoader upload are left out (before, only those of the Sources tab).


In [1]:
import re
import time
from datetime import datetime, timezone

from delta.exceptions import DeltaConcurrentModificationException
from pyspark.sql import functions as F
from pyspark.sql import types as T


# Grain roles: physical column declared in the Sources tab -> standard name used in
# the grouping key (key_json_ln). An empty role is not part of the grouping key.
# grp_column_list_ln is only used by rows created before the Sources tab existed.
GRAIN_ROLES = (
    ("ent_column_nm", "ENTIDADE"),
    ("ident_column_nm", "IDENT"),
    ("subent_column_nm", "IDSUBENT"),
    ("ctry_column_nm", "CDWA"),
)


def grain_internal_column(key):
    """Internal name of a grain column in the working DataFrame. It never overwrites a
    physical column of the source table (filters and metrics may use them)."""
    return f"__dpm_grain_{key}"


def load_source_config(frame):
    """Source table configuration of the given Sources rows.

    Each parameterisation file declares its own source tables: the key is
    (file, src_tbl_id) and each datapoint uses the Sources of its own file
    (source_config_for). Rows without a file were migrated from the configuration that
    existed before the Sources tab and are only used to re-run earlier snapshots.
    """
    available = {name.lower() for name in frame.columns}
    role_mode = any(column in available for column, _ in GRAIN_ROLES)
    rows = frame_records(frame)
    config = {}
    for row in rows:
        if role_mode:
            grain_columns = tuple(
                (standard, clean(row.get(column)).upper())
                for column, standard in GRAIN_ROLES
            )
        else:
            grain_columns = tuple((name, name) for name in split_csv(row.get("grp_column_list_ln")))
        logical = clean(row["src_tbl_id"]).upper()
        config[(clean(row.get("src_file_nm")), logical)] = {
            "src_file_nm": clean(row.get("src_file_nm")),
            "src_tbl_id": logical,
            "phy_tbl_nm": clean(row["phy_tbl_nm"]),
            "per_column_nm": clean(row["per_column_nm"]),
            # Period format: DATE or a pattern made of yyyy, MM and dd. Empty only in rows
            # created before the Sources tab (format inferred from the data type).
            "per_fmt_cd": normalize_period_format(row.get("per_fmt_cd")),
            # (name in key_json_ln, physical column or "" if the role is not part of the key)
            "grain_columns": grain_columns,
            "group_columns": tuple(key for key, _ in grain_columns),
            "grain_id": clean(row.get("grain_id")) or logical,
        }
    return config


def source_config_for(source_file, logical_source):
    """Sources of the datapoint's own file; otherwise, the migrated row without a file."""
    logical = clean(logical_source).upper()
    return SOURCE_CONFIG.get((clean(source_file), logical)) or SOURCE_CONFIG.get(("", logical))


# Assigned in the run cell; declared here so that later cells can run in isolation.
SOURCE_CONFIG = {}


DECIMAL_SQL = "DECIMAL(38,10)"


ALLOWED_AGGREGATIONS = {"SUM", "COUNT", "MIN", "MAX", "AVG", "FIRST"}


NULL_TOKENS = {"NULL", "<NULL>", "."}   # "." = null token inherited from the previous solution


LOCAL_VARIABLE_PATTERN = re.compile(r"\bV\d+\b", flags=re.I)


def utc_now():
    """Current time in UTC, without time zone (naive)."""
    return datetime.now(timezone.utc).replace(tzinfo=None)


def clean(value):
    """Value as stripped text; None and 'nan' become an empty string."""
    if value is None:
        return ""
    text = str(value).strip()
    return "" if text.lower() == "nan" else text


def frame_records(frame):
    """Rows of a small DataFrame as dictionaries, with nulls as empty strings."""
    return [
        {name: "" if value is None else value for name, value in row.asDict().items()}
        for row in frame.toLocalIterator()
    ]


def split_csv(value):
    """Comma- or semicolon-separated values in upper case, without blanks or duplicates."""
    return tuple(
        dict.fromkeys(
            item.strip().upper()
            for item in clean(value).replace(";", ",").split(",")
            if item.strip()
        )
    )


def quote_identifier(value):
    """Identifier between backticks, with inner backticks escaped."""
    return "`" + str(value).replace("`", "``") + "`"


def sql_literal(value):
    """SQL string literal, with inner single quotes escaped."""
    return "'" + str(value).replace("'", "''") + "'"


def validate_reference_period(value):
    """Checks a reference period in YYYYMM or YYYYMMDD format and returns it cleaned."""
    value = clean(value)
    if re.fullmatch(r"\d{6}", value):
        if not 1 <= int(value[4:6]) <= 12:
            raise ValueError(f"reference_period month must be 01-12; received {value!r}")
        return value
    if re.fullmatch(r"\d{8}", value):
        if not 1 <= int(value[4:6]) <= 12 or not 1 <= int(value[6:8]) <= 31:
            raise ValueError(f"reference_period day must be 01-31; received {value!r}")
        return value
    raise ValueError(f"reference_period must use YYYYMM or YYYYMMDD; received {value!r}")


def add_months(yyyymm, offset):
    """Month YYYYMM shifted by `offset` months."""
    yyyymm = validate_reference_period(yyyymm)
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    absolute = year * 12 + month - 1 + int(offset)
    return f"{absolute // 12:04d}{absolute % 12 + 1:02d}"


class UnsupportedPeriod(ValueError):
    """Unsupported period value: raises META.PERIOD.INVALID."""


# Period syntax (relative to the reference period of the run):
#   M / Execution Month / Current / empty -> reference month
#   M-n (n = 0..99)                       -> n months before the reference month
#   Y/MM                                  -> month MM of the reference year
#   Y-k/MM (k = 0..9)                     -> month MM of k years before
PERIOD_CURRENT_LABELS = {"", "M", "CURRENT", "CURRENTMONTH", "EXECUTIONMONTH"}


PERIOD_RELATIVE_PATTERN = re.compile(r"M-(\d{1,2})")


PERIOD_ANCHORED_PATTERN = re.compile(r"Y(?:-(\d))?/(0[1-9]|1[0-2])")


def resolve_period(reference_period_value, label):
    """Physical month and period codes of a period expression (M, M-n, Y/MM, Y-k/MM)."""
    text = re.sub(r"\s+", "", clean(label).upper())
    relative = PERIOD_RELATIVE_PATTERN.fullmatch(text)
    anchored = PERIOD_ANCHORED_PATTERN.fullmatch(text)
    if text in PERIOD_CURRENT_LABELS or (relative and int(relative.group(1)) == 0):
        # M-0 is normalised to M: same month, same read group.
        month = reference_period_value
    elif relative:
        month = add_months(reference_period_value, -int(relative.group(1)))
    elif anchored:
        year = int(reference_period_value[:4]) - int(anchored.group(1) or 0)
        month = f"{year:04d}{anchored.group(2)}"
    else:
        raise UnsupportedPeriod(f"Unsupported period value: {clean(label)!r}")
    return {
        "physical_month": month,
        "per_cd": f"{month[:4]}M{month[4:6]}",
        "per_frq_cd": "M",
    }


# ---------------------------------------------------------------------------
# Period format of each source table.
#   DATE     -> date/timestamp column, compared as a date.
#   pattern  -> text or number read with a pattern made of yyyy, MM and optionally dd,
#               separated by - / . _ or space (yyyyMM, yyyy-MM-dd...). Without dd = last
#               day of the month.
#   empty    -> only in rows created before the Sources tab: inferred from the data type.
# A value is only accepted if rewriting it with the pattern yields exactly the original
# text (strict reading), so the lineage query can filter on the text (periodo = '202512').
# ---------------------------------------------------------------------------
PERIOD_FORMAT_DATE = "DATE"


PERIOD_FORMAT_TOKEN = re.compile(r"yyyy|MM|dd|[-/._ ]")


PERIOD_FORMAT_PYTHON = {"yyyy": "%Y", "MM": "%m", "dd": "%d"}


def normalize_period_format(value):
    """Period format as declared; DATE in any case becomes DATE."""
    text = clean(value)
    return PERIOD_FORMAT_DATE if text.upper() == PERIOD_FORMAT_DATE else text


def period_format_tokens(period_format):
    """Tokens of a valid pattern, or None if the pattern is not valid."""
    fmt = normalize_period_format(period_format)
    tokens, pos = [], 0
    while pos < len(fmt):
        match = PERIOD_FORMAT_TOKEN.match(fmt, pos)
        if not match:
            return None
        tokens.append(match.group(0))
        pos = match.end()
    if tokens.count("yyyy") != 1 or tokens.count("MM") != 1 or tokens.count("dd") > 1:
        return None
    return tokens


def period_format_problem(period_format, data_type):
    """'' if the format is valid for the column; otherwise, the reason (logged)."""
    fmt = normalize_period_format(period_format)
    name = data_type.simpleString().split("(")[0].lower()
    is_date = name in {"date", "timestamp"}
    if fmt == PERIOD_FORMAT_DATE:
        return "" if is_date else f"DATE requires a date or timestamp column; the column is {name}"
    if period_format_tokens(fmt) is None:
        return (
            "use DATE or a pattern made of yyyy, MM and optionally dd, "
            "separated by - / . _ or space (e.g. yyyyMM, yyyy-MM-dd)"
        )
    if is_date:
        return f"the column is {name}: use DATE"
    return ""


def period_has_day(period_format):
    """True if the period format includes the day (dd)."""
    return "dd" in (period_format_tokens(period_format) or [])


def format_period_value(moment, period_format):
    """Date -> text in the format of the table (used by the lineage query)."""
    return "".join(
        moment.strftime(PERIOD_FORMAT_PYTHON[token]) if token in PERIOD_FORMAT_PYTHON else token
        for token in period_format_tokens(period_format)
    )


def period_text_expr(column):
    """Period column as trimmed text."""
    return F.trim(F.col(quote_identifier(column)).cast("string"))


def period_parsed_expr(column, period_format):
    """Date parsed with the pattern, only if the parsing is exact; otherwise NULL."""
    text = period_text_expr(column)
    parsed = F.to_date(text, period_format)
    return F.when(parsed.isNotNull() & (F.date_format(parsed, period_format) == text), parsed)


def period_to_date_expr(column, data_type, period_format=""):
    """Period column as a date, read with its declared format (or its data type if none)."""
    fmt = normalize_period_format(period_format)
    name = data_type.simpleString().split("(")[0].lower()
    if fmt == PERIOD_FORMAT_DATE or (not fmt and name in {"date", "timestamp"}):
        return F.col(quote_identifier(column)).cast("date")
    if fmt:
        parsed = period_parsed_expr(column, fmt)
        return parsed if period_has_day(fmt) else F.last_day(parsed)
    # No declared format (rows created before the Sources tab): inferred from the digits.
    text = F.regexp_replace(F.col(quote_identifier(column)).cast("string"), "[^0-9]", "")
    return (
        F.when(F.length(text) == 6, F.last_day(F.to_date(text, "yyyyMM")))
        .when(F.length(text) == 8, F.to_date(text, "yyyyMMdd"))
        .otherwise(F.lit(None).cast("date"))
    )


def period_unparseable_expr(column, period_format):
    """Rows with a non-empty period that the declared format cannot read."""
    fmt = normalize_period_format(period_format)
    if not fmt or fmt == PERIOD_FORMAT_DATE:
        return F.lit(False)
    return F.col(quote_identifier(column)).isNotNull() & period_parsed_expr(column, fmt).isNull()


def period_literal_to_date(value):
    """Reference period (YYYYMM or YYYYMMDD) as a date; YYYYMM is the last day of the month."""
    digits = re.sub(r"\D", "", clean(value))
    if len(digits) == 6:
        return F.last_day(F.to_date(F.lit(digits), "yyyyMM"))
    if len(digits) == 8:
        return F.to_date(F.lit(digits), "yyyyMMdd")
    raise ValueError(f"Unsupported reference period: {value!r}")


def parse_as_of(value):
    """Timestamp requested in metadata_as_of, in UTC; None when empty or 'latest'."""
    text = clean(value)
    if not text or text.lower() == "latest":
        return None
    parsed = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if parsed.tzinfo is not None:
        parsed = parsed.astimezone(timezone.utc).replace(tzinfo=None)
    return parsed


def has_scd_contract(frame):
    """True if the table has the SCD 2.2 columns needed to read a snapshot."""
    return {"ref_start_dt", "ref_end_dt", "row_present_flg"}.issubset(set(frame.columns))


def latest_scd_timestamp(table_name):
    """Latest version change (start or end) among the present rows of an SCD 2.2 table, so
    that a load that only removes rows also creates a new snapshot."""
    frame = spark.table(table_name)
    if not has_scd_contract(frame):
        raise RuntimeError(f"Table {table_name} is not SCD Type 2.2 compatible.")
    row = (
        frame.where(F.col("row_present_flg") == 1)
        .agg(F.greatest(F.max("ref_start_dt"), F.max("ref_end_dt")).alias("ts"))
        .first()
    )
    if not row or row["ts"] is None:
        raise ValueError(f"No SCD rows are available in {table_name}.")
    return row["ts"]


def resolve_metadata_as_of(requested):
    """Parameterisation snapshot to use: the requested one, or the latest of the parameterisation tables."""
    explicit = parse_as_of(requested)
    if explicit is not None:
        return explicit
    return max(
        latest_scd_timestamp(datapoints_table),
        latest_scd_timestamp(filters_table),
        latest_scd_timestamp(catalog_table),
        # The source configuration is part of the snapshot, so `latest` includes it.
        latest_scd_timestamp(source_config_table),
    )


def scd_snapshot(table_name, as_of):
    """Rows of an SCD 2.2 table valid at `as_of` and not logically deleted."""
    frame = spark.table(table_name)
    if not has_scd_contract(frame):
        raise RuntimeError(
            f"Table {table_name} does not expose ref_start_dt/ref_end_dt/row_present_flg."
        )
    return frame.where(
        (F.col("ref_start_dt") <= F.lit(as_of))
        & (F.col("ref_end_dt").isNull() | (F.col("ref_end_dt") > F.lit(as_of)))
        & (F.col("row_present_flg") == 1)
    )


# ---------------------------------------------------------------------------
# Filter values (Values column of the Filters tab).
#   - Single-value operators (=, >, >=, <, <=): the whole text is ONE value, commas
#     included: `= T02.11,i0190` selects the code 'T02.11,i0190'.
#   - IN, NOT IN, LIKE, NOT LIKE and BETWEEN: the values are separated by commas. A
#     value that contains a comma is written between double quotes:
#     `IN "T02.11,i0190","T02.11,i0220"`. Inside quotes, "" stands for one quote.
#   - Unquoted values keep the previous behaviour: surrounding ' or " are removed.
# Double quotes, not single ones: Excel hides an apostrophe at the start of a cell.
# ---------------------------------------------------------------------------
SINGLE_VALUE_OPERATORS = {"=", "EQUAL", "EQUALS", ">", ">=", "<", "<="}


class FilterValueError(ValueError):
    """Values text that cannot be read (META.FILTER.VALUE_SYNTAX_INVALID)."""


def _split_filter_values(text):
    """Comma-separated values; a value between double quotes may contain commas."""
    values, pos, length = [], 0, len(text)
    while True:
        while pos < length and text[pos].isspace():
            pos += 1
        if pos < length and text[pos] == '"':
            pos += 1
            chars = []
            while True:
                if pos >= length:
                    raise FilterValueError("a double quote is not closed")
                if text[pos] == '"':
                    if pos + 1 < length and text[pos + 1] == '"':
                        chars.append('"')
                        pos += 2
                        continue
                    pos += 1
                    break
                chars.append(text[pos])
                pos += 1
            value = "".join(chars).strip()
            while pos < length and text[pos].isspace():
                pos += 1
            if pos < length and text[pos] != ",":
                raise FilterValueError("text after a closing double quote; separate the values with commas")
        else:
            end = text.find(",", pos)
            end = length if end < 0 else end
            value = text[pos:end].strip().strip("'\"")
            pos = end
        if value:
            values.append(value)
        if pos >= length:
            return tuple(values)
        pos += 1  # comma


def filter_value_tokens(raw, operator=""):
    """Values of a filter, read according to its operator (see the block header)."""
    text = clean(raw)
    if not text:
        return ()
    if clean(operator).upper() in SINGLE_VALUE_OPERATORS and not text.startswith('"'):
        value = text.strip("'\"").strip()
        return (value,) if value else ()
    return _split_filter_values(text)


def filter_type_family(data_type):
    """Type family used to build filters: numeric, date, timestamp or string."""
    name = data_type.simpleString().split("(")[0].lower()
    # simpleString() returns tinyint/smallint/int/bigint for integer columns; without them
    # range filters would compare text ('99' > '100').
    if name in {
        "tinyint", "smallint", "int", "bigint",
        "byte", "short", "integer", "long",
        "float", "double", "decimal",
    }:
        return "numeric"
    if name in {"date", "timestamp"}:
        return name
    return "string"


def build_filter_sql(item, source_schema):
    """SQL condition of one filter row, adapted to the type of the source column."""
    column = clean(item.get("src_column_nm"))
    operator = clean(item.get("optr_cd")).upper()
    tokens = filter_value_tokens(item.get("fltr_val_ln"), operator)
    actual = next((name for name in source_schema if name.upper() == column.upper()), column)
    family = filter_type_family(source_schema[actual]) if actual in source_schema else "string"
    null_requested = any(token.upper() in NULL_TOKENS for token in tokens)
    values = [token for token in tokens if token.upper() not in NULL_TOKENS]

    qcol = quote_identifier(actual)
    if family == "string":
        left = f"TRIM(UPPER(CAST({qcol} AS STRING)))"
        rendered_values = [value.upper() for value in values]
        literal = sql_literal
    elif family == "numeric":
        left = f"CAST({qcol} AS {DECIMAL_SQL})"
        rendered_values = values
        literal = lambda value: f"CAST({sql_literal(value)} AS {DECIMAL_SQL})"
    else:
        left = qcol
        # A date written without dashes (20240101) is normalised to 2024-01-01.
        rendered_values = [
            f"{value[:4]}-{value[4:6]}-{value[6:8]}" if re.fullmatch(r"\d{8}", value) else value
            for value in values
        ]
        literal = sql_literal

    if operator in {"=", "EQUAL", "EQUALS", "IN"}:
        parts = []
        if rendered_values:
            if operator != "IN" and len(rendered_values) == 1:
                parts.append(f"{left} = {literal(rendered_values[0])}")
            else:
                parts.append(
                    f"{left} IN ({', '.join(literal(value) for value in rendered_values)})"
                )
        if null_requested:
            parts.append(f"{qcol} IS NULL")
        return "(" + " OR ".join(parts) + ")"

    if operator in {"NOT IN", "NOTIN"}:
        parts = []
        if rendered_values:
            parts.append(
                f"{left} NOT IN ({', '.join(literal(value) for value in rendered_values)})"
            )
        if null_requested:
            parts.append(f"{qcol} IS NOT NULL")
        return "(" + " AND ".join(parts) + ")"

    # LIKE compares text. Numeric columns use their own text representation, not the
    # DECIMAL(38,10) one, which adds trailing zeros and breaks patterns such as '%5'.
    like_left = f"TRIM(UPPER(CAST({qcol} AS STRING)))" if family == "numeric" else left

    if operator == "LIKE":
        parts = [f"{like_left} LIKE {sql_literal(value.upper())}" for value in rendered_values]
        if null_requested:
            parts.append(f"{qcol} IS NULL")
        return "(" + " OR ".join(parts) + ")"

    if operator in {"NOT LIKE", "NOTLIKE"}:
        parts = [f"{like_left} NOT LIKE {sql_literal(value.upper())}" for value in rendered_values]
        if null_requested:
            parts.append(f"{qcol} IS NOT NULL")
        return "(" + " AND ".join(parts) + ")"

    if operator in {">", ">=", "<", "<="}:
        # Range operators: a null value is compared as 0 in numeric columns and as
        # 1900-01-01 in date/timestamp columns.
        if family == "numeric":
            base = f"COALESCE(CAST({qcol} AS {DECIMAL_SQL}), CAST(0 AS {DECIMAL_SQL}))"
        elif family in {"date", "timestamp"}:
            base = f"COALESCE({qcol}, DATE'1900-01-01')"
        else:
            base = left
        return "(" + " AND ".join(
            f"{base} {operator} {literal(v)}" for v in rendered_values
        ) + ")"

    if operator == "BETWEEN" and len(rendered_values) == 2:
        if family == "numeric":
            base = f"COALESCE(CAST({qcol} AS {DECIMAL_SQL}), CAST(0 AS {DECIMAL_SQL}))"
        elif family in {"date", "timestamp"}:
            base = f"COALESCE({qcol}, DATE'1900-01-01')"
        else:
            base = left
        return f"({base} BETWEEN {literal(rendered_values[0])} AND {literal(rendered_values[1])})"

    raise RuntimeError(f"Validated filter operator became invalid at runtime: {operator!r}")


def prepare_grain_columns(source, grain_columns):
    """Add the internal grain columns.

    Each role is read from its physical column (case-insensitive). If the table does not
    have that column, or the role is not declared, the internal column is NULL and is
    not used for grouping.
    """
    existing = {name.upper(): name for name in source.columns}
    for key, physical in grain_columns:
        actual = existing.get(clean(physical).upper()) if clean(physical) else None
        source = source.withColumn(
            grain_internal_column(key),
            F.col(quote_identifier(actual)) if actual else F.lit(None).cast("string"),
        )
    return source


def parse_rollup_expression(expression):
    """Split a Rollup expression into (child datapoint ID, sign).

    Every + or - outside square brackets is an operator; the text between two operators
    is a child datapoint ID, whatever its format. An ID fully wrapped in brackets has them
    removed. Spaces around the operators are optional.
      'var:[A] + var:[B]_geo_LU' -> [('var:[A]', 1), ('var:[B]_geo_LU', 1)]
      '[F 01.01.r020.c010] - [F 01.01.r030.c010]' -> [('F 01.01.r020.c010', 1), ('F 01.01.r030.c010', -1)]
    """
    text = clean(expression)
    terms, current, sign, depth = [], [], 1, 0

    def close_term():
        """Adds the text read so far as a Rollup term, with its sign."""
        child = clean("".join(current))
        if re.fullmatch(r"\[[^\[\]]+\]", child):
            child = clean(child[1:-1])
        if not child:
            raise ValueError(f"Invalid rollup expression: {expression!r}")
        terms.append((child, sign))

    for char in text:
        if char == "[":
            depth += 1
        elif char == "]":
            depth -= 1
        if char in "+-" and depth == 0:
            if clean("".join(current)):
                close_term()
            elif terms:
                # Two operators in a row (e.g. "A + - B").
                raise ValueError(f"Invalid rollup expression: {expression!r}")
            current, sign = [], (-1 if char == "-" else 1)
            continue
        current.append(char)

    if depth != 0:
        raise ValueError(f"Invalid rollup expression: {expression!r}")
    close_term()
    return terms


# Value type of the datapoint (Value Type column of the Catalog tab). It only
# determines the publication format; the calculation is always DECIMAL(38,10).
VALUE_TYPES = {"AMOUNT", "PERCENTAGE", "INTEGER", "TEXT"}


def upload_rows(table_name, dataload_id):
    """Rows of one DataLoader load of a raw table, with their text values trimmed (as they
    reach the parameterisation tables)."""
    frame = spark.table(table_name).where(F.col("dataload_id") == dataload_id)
    return frame.select([
        F.trim(F.col(field.name)).alias(field.name)
        if isinstance(field.dataType, T.StringType) else F.col(field.name)
        for field in frame.schema.fields
    ])


# Columns that do not make a row of a tab non-blank: the ones the DataLoader adds to every
# raw table, the file name and the Check column of the tab.
RAW_NON_CONTENT_COLUMNS = {
    "dataload_id", "ingestion_timestamp", "usr_id", "src_file_nm", "excel_check_ln",
    "load_dt", "metadata_load_id",
}


def without_blank_rows(frame):
    """Rows of a tab with some content. The tabs of the file are formatted for more rows than
    they use, and the DataLoader also loads those empty rows."""
    content = [name for name in frame.columns if name.lower() not in RAW_NON_CONTENT_COLUMNS]
    return frame.where(F.length(F.concat_ws("", *[F.col(name).cast("string") for name in content])) > 0)


def candidate_frames(candidate):
    """Datapoints, filters, catalog and sources of one DataLoader upload, read from the raw
    tables by the dataload_id of each tab. Blank rows of the tabs are left out."""
    raw_tables = {
        "datapoints": raw_datapoints_table,
        "filters": raw_filters_table,
        "catalog": raw_catalog_table,
        "sources": raw_source_config_table,
    }
    not_informed = sorted(name for name in raw_tables if not clean(candidate.get(name)))
    if not_informed:
        raise ValueError(f"candidate_json has no dataload_id for: {not_informed}")
    return tuple(
        without_blank_rows(upload_rows(table_name, clean(candidate[name])))
        for name, table_name in raw_tables.items()
    )


# A MERGE or a replaceWhere on a table without partitions is rejected by Delta when
# another run commits to the same table at the same time. The write is then repeated.
WRITE_ATTEMPTS = 3
WRITE_RETRY_WAIT_SECONDS = 30


def write_with_retry(write):
    """Runs a Delta write. If another run has committed to the same table meanwhile, waits
    and repeats it on the new version of the table; the last attempt raises the conflict."""
    for attempt in range(1, WRITE_ATTEMPTS):
        try:
            return write()
        except DeltaConcurrentModificationException:
            time.sleep(WRITE_RETRY_WAIT_SECONDS * attempt)
    return write()


StatementMeta(, 07087ef3-e8e7-42b2-be61-500440875cfd, 5, Finished, Available, Finished, False)